# Import libraries

In [1]:
import pandas as pd
import os
import shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import threading
from typing import Tuple
import json 
from tqdm import tqdm

import sys
sys.path.append('..')
from utils.audio_util import convert_mp3_to_flac, resample_audios, trim_silence_with_vad, normalize_audio_files
from utils.file_util import recursive_copy
from utils.text_util import clean_text_cv, handle_maiyamok

from transformers import Wav2Vec2FeatureExtractor, WavLMForXVector
import torch
import torchaudio
from collections import defaultdict
import gc
import numpy as np
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_distances

# Moving files to new directory

In [2]:
df = pd.read_csv("../data/raw/thai-central/thai-central_mapping.csv")

In [3]:
AUDIO_BASE_DIR = "../data/raw/thai-central/audio_v2"
DEST_DIR = "../data/converted/thai-central-to-virtual-vctk"
AUDIO_DEST_DIR = os.path.join(DEST_DIR, "wav16")
TXT_DEST_DIR = os.path.join(DEST_DIR, "txt")

In [4]:
# Add full path column
df['full_path'] = df['public_name'].apply(lambda x: os.path.join(AUDIO_BASE_DIR, x))

# Filter existing files
df_filtered = df[df['full_path'].apply(os.path.exists)].copy()

# Count files per speaker
speaker_counts = df_filtered['speaker_id'].value_counts()
valid_speakers = speaker_counts[speaker_counts >= 100].index

# Filter speakers with >= 100 files
df_filtered = df_filtered[df_filtered['speaker_id'].isin(valid_speakers)]
df_filtered['key'] = df_filtered['public_name'].apply(lambda x: x.split("/")[-1].split(".")[0].split("_")[1])

In [5]:
df_filtered

,speaker_id,original_name,public_name,full_path,key
3,spk-560353791001250275300,60f7422cf22f3d24deb858fd_1627814081674.wav,train_audio02/thai-central_000003.mp3,../data/raw/thai-central/audio_v2/train_audio0...,000003
5,spk8789214662434341888000,611bdfa7cf5abe28e3190dab_1637558034204.wav,train_audio12/thai-central_000005.mp3,../data/raw/thai-central/audio_v2/train_audio1...,000005
6,spk2272676593238898160000,6107d2adfb309b3360224bb3_1637129050946.wav,train_audio06/thai-central_000006.mp3,../data/raw/thai-central/audio_v2/train_audio0...,000006
7,spk-659303002028011164800,60f7422cf22f3d24deb85922_1628183509033.wav,train_audio01/thai-central_000007.mp3,../data/raw/thai-central/audio_v2/train_audio0...,000007
8,spk5308879110509872434000,60f7422cf22f3d24deb858ff_1628702261034.wav,train_audio02/thai-central_000008.mp3,../data/raw/thai-central/audio_v2/train_audio0...,000008
...,...,...,...,...,...
433806,spk-717737201021391723000,611be1b6cf5abe28e3191fe0_1636618869118.wav,train_audio05/thai-central_433806.mp3,../data/raw/thai-central/audio_v2/train_audio0...,433806
433808,spk8339279944661639402000,60f7422cf22f3d24deb85917_1629169814235.wav,train_audio02/thai-central_433808.mp3,../data/raw/thai-central/audio_v2/train_audio0...,433808
433811,spk7617813596557544310000,6107d2adfb309b3360224ba8_1637550218939.wav,train_audio12/thai-central_433811.mp3,../data/raw/thai-central/audio_v2/train_audio1...,433811
433812,spk-581523479174861464600,611be1b6cf5abe28e3191fd3_1629719575649.wav,train_audio04/thai-central_433812.mp3,../data/raw/thai-central/audio_v2/train_audio0...,433812


In [6]:
print(f"There are {len(df_filtered)} samples in the dataset.")

There are 225028 samples in the dataset.


In [7]:
headers = ['filename', 'txt', 'transcript', 'cer']
df_cer = pd.read_csv("../analysis/transcribed_refined.csv", delimiter="|", names=headers)

df_cer['cer'].apply(int)
df_cer['key'] = df_cer['filename'].apply(lambda x: x.split("/")[-1].split(".")[0].split("_")[1])
df_cer = df_cer[['key', 'cer']]
df_cer

,key,cer
0,201353,0.543210
1,077852,0.070423
2,216452,0.076471
3,040893,0.000000
4,070804,0.055556
...,...,...
225023,140591,0.789474
225024,006202,0.031250
225025,252112,0.550000
225026,243408,0.000000


In [8]:
threshold = 0.1

df_filtered = df_filtered.merge(df_cer, on='key', how='left')
df_filtered = df_filtered[df_filtered['cer'] <= threshold]
df_filtered


,speaker_id,original_name,public_name,full_path,key,cer
1,spk8789214662434341888000,611bdfa7cf5abe28e3190dab_1637558034204.wav,train_audio12/thai-central_000005.mp3,../data/raw/thai-central/audio_v2/train_audio1...,000005,0.041667
2,spk2272676593238898160000,6107d2adfb309b3360224bb3_1637129050946.wav,train_audio06/thai-central_000006.mp3,../data/raw/thai-central/audio_v2/train_audio0...,000006,0.033333
3,spk-659303002028011164800,60f7422cf22f3d24deb85922_1628183509033.wav,train_audio01/thai-central_000007.mp3,../data/raw/thai-central/audio_v2/train_audio0...,000007,0.000000
5,spk4556132968060195345000,611bdfa7cf5abe28e3190dae_1637638572891.wav,train_audio09/thai-central_000012.mp3,../data/raw/thai-central/audio_v2/train_audio0...,000012,0.032258
7,spk-629673445020237031000,6107d2adfb309b3360224bd0_1633674000446.wav,train_audio04/thai-central_000014.mp3,../data/raw/thai-central/audio_v2/train_audio0...,000014,0.045455
...,...,...,...,...,...,...
225018,spk-488486316582704303900,6107d2adfb309b3360224bb9_1632632911566.wav,train_audio03/thai-central_433798.mp3,../data/raw/thai-central/audio_v2/train_audio0...,433798,0.000000
225022,spk8856238053062916626000,611bdfa7cf5abe28e3190dad_1637223711972.wav,train_audio08/thai-central_433805.mp3,../data/raw/thai-central/audio_v2/train_audio0...,433805,0.032680
225023,spk-717737201021391723000,611be1b6cf5abe28e3191fe0_1636618869118.wav,train_audio05/thai-central_433806.mp3,../data/raw/thai-central/audio_v2/train_audio0...,433806,0.000000
225025,spk7617813596557544310000,6107d2adfb309b3360224ba8_1637550218939.wav,train_audio12/thai-central_433811.mp3,../data/raw/thai-central/audio_v2/train_audio1...,433811,0.051546


In [9]:
print(f"There are {len(df_filtered)} samples in the dataset with CER >= {threshold}.")

There are 99325 samples in the dataset with CER >= 0.1.


In [10]:
# Create new speaker ID mapping
sorted_speakers = speaker_counts[speaker_counts >= 100].sort_values().index
speaker_mapping = {
    spk: f'tc{i+1:04d}' 
    for i, spk in enumerate(sorted_speakers)
}

# Add new speaker ID column
df_filtered['new_speaker_id'] = df_filtered['speaker_id'].map(speaker_mapping)
df_filtered

,speaker_id,original_name,public_name,full_path,key,cer,new_speaker_id
1,spk8789214662434341888000,611bdfa7cf5abe28e3190dab_1637558034204.wav,train_audio12/thai-central_000005.mp3,../data/raw/thai-central/audio_v2/train_audio1...,000005,0.041667,tc0675
2,spk2272676593238898160000,6107d2adfb309b3360224bb3_1637129050946.wav,train_audio06/thai-central_000006.mp3,../data/raw/thai-central/audio_v2/train_audio0...,000006,0.033333,tc0571
3,spk-659303002028011164800,60f7422cf22f3d24deb85922_1628183509033.wav,train_audio01/thai-central_000007.mp3,../data/raw/thai-central/audio_v2/train_audio0...,000007,0.000000,tc0898
5,spk4556132968060195345000,611bdfa7cf5abe28e3190dae_1637638572891.wav,train_audio09/thai-central_000012.mp3,../data/raw/thai-central/audio_v2/train_audio0...,000012,0.032258,tc0251
7,spk-629673445020237031000,6107d2adfb309b3360224bd0_1633674000446.wav,train_audio04/thai-central_000014.mp3,../data/raw/thai-central/audio_v2/train_audio0...,000014,0.045455,tc0448
...,...,...,...,...,...,...,...
225018,spk-488486316582704303900,6107d2adfb309b3360224bb9_1632632911566.wav,train_audio03/thai-central_433798.mp3,../data/raw/thai-central/audio_v2/train_audio0...,433798,0.000000,tc0957
225022,spk8856238053062916626000,611bdfa7cf5abe28e3190dad_1637223711972.wav,train_audio08/thai-central_433805.mp3,../data/raw/thai-central/audio_v2/train_audio0...,433805,0.032680,tc0494
225023,spk-717737201021391723000,611be1b6cf5abe28e3191fe0_1636618869118.wav,train_audio05/thai-central_433806.mp3,../data/raw/thai-central/audio_v2/train_audio0...,433806,0.000000,tc0649
225025,spk7617813596557544310000,6107d2adfb309b3360224ba8_1637550218939.wav,train_audio12/thai-central_433811.mp3,../data/raw/thai-central/audio_v2/train_audio1...,433811,0.051546,tc0656


In [11]:
df_train = pd.read_csv("../data/raw/thai-central/train.csv")
df_dev = pd.read_csv("../data/raw/thai-central/dev.csv")

df_all = pd.concat([df_train, df_dev], ignore_index=True)
df_all['sentence'] = df_all['sentence'].apply(lambda x: "".join(x.split()))
df_all['audio'] = df_all['audio'].apply(lambda x: os.path.join(AUDIO_BASE_DIR, x))
df_all

,utterance,sentence,audio
0,thai-central_000000,ทีมจากอิสราเอลไม่ควรได้เป็นเจ้าบ้านในเกมยูฟ่าคัพ,../data/raw/thai-central/audio_v2/train_audio1...
1,thai-central_000001,แต่พอไหมอะไรคือแต้อีบ็อบฮ่าฮ่ากูพิมพ์ผิดไหมล่ะ...,../data/raw/thai-central/audio_v2/train_audio0...
2,thai-central_000003,ทุกสิ่งทุกอย่างจะราบรื่น,../data/raw/thai-central/audio_v2/train_audio0...
3,thai-central_000005,เร็วหันมองเวลาตั้งกระทู้,../data/raw/thai-central/audio_v2/train_audio1...
4,thai-central_000006,มีขนาดหนาและใหญ่กว่าเกร็ดปลาทั่วไปจนเหมือนเครื...,../data/raw/thai-central/audio_v2/train_audio0...
...,...,...,...
341134,thai-central_433292,มีของทั้งหมดเป็นจำนวนหนึ่งหมื่นหนึ่งพันกระป๋องค่ะ,../data/raw/thai-central/audio_v2/dev_audio00/...
341135,thai-central_433304,บ้านงิ้วงามหมู่สี่มีอาณาเขตติดต่อกับหมู่บ้านใก...,../data/raw/thai-central/audio_v2/dev_audio00/...
341136,thai-central_433457,กองทัพเรือหมายถึงกองกำลังทางทหารที่ปฏิบัติการท...,../data/raw/thai-central/audio_v2/dev_audio00/...
341137,thai-central_433700,กรมอู่ทหารเรือ,../data/raw/thai-central/audio_v2/dev_audio00/...


In [12]:
audio2sentence = dict(zip(df_all['audio'], df_all['sentence']))

In [13]:
# Thread-safe set for character collection
all_chars = set()
chars_lock = threading.Lock()

# Thread-safe list for tracking skipped files
skip_files = []
skip_lock = threading.Lock()

def process_file_pair(args: Tuple[str, str, str, str]) -> None:
    """Process a single pair of audio and text files"""
    speaker_id, src_path, dest_audio_path, dest_txt_path = args
    try:
        # Create speaker directories
        speaker_wav_dir = os.path.join(AUDIO_DEST_DIR, speaker_id)
        speaker_txt_dir = os.path.join(TXT_DEST_DIR, speaker_id)
        os.makedirs(speaker_wav_dir, exist_ok=True)
        os.makedirs(speaker_txt_dir, exist_ok=True)
        
        # Process audio
        dest_filename = os.path.splitext(os.path.basename(dest_audio_path))[0] + '_mic1.flac'
        dest_path = os.path.join(speaker_wav_dir, dest_filename)
        
        if not convert_mp3_to_flac(src_path, dest_path):
            raise Exception("Failed to convert audio")
        
        # Create empty text file and collect characters
        base_filename = os.path.splitext(dest_filename)[0]
        txt_filename = f"{base_filename}.txt"
        txt_path = os.path.join(speaker_txt_dir, txt_filename)
        
        # In this case we're creating empty text files
        # Modify this part if you need to process actual text content
        with open(txt_path, 'w', encoding='utf-8') as f:
            f.write(audio2sentence[src_path])

        # Collect characters
        with chars_lock:
            all_chars.update(audio2sentence[src_path])
            
    except Exception as e:
        print(f"Error processing file {src_path}: {e}")
        with skip_lock:
            skip_files.append(src_path)

# Remove existing directories if they exist
if os.path.exists(DEST_DIR):
    print("Clearing destination folder")
    shutil.rmtree(DEST_DIR)

# Create necessary directories
os.makedirs(AUDIO_DEST_DIR, exist_ok=True)
os.makedirs(TXT_DEST_DIR, exist_ok=True)

# Create processing arguments
process_args = [
    (row['new_speaker_id'], row['full_path'], 
        os.path.join(AUDIO_DEST_DIR, row['new_speaker_id'], os.path.basename(row['public_name'])),
        os.path.join(TXT_DEST_DIR, row['new_speaker_id'], os.path.basename(row['public_name'])))
    for _, row in df_filtered.iterrows()
]

# Process files in parallel with progress bar
max_workers = os.cpu_count()
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    list(tqdm(
        executor.map(process_file_pair, process_args),
        total=len(process_args),
        desc=f"Processing files (using {max_workers} workers)"
    ))

# Print results
print(f"Processed {len(df_filtered) - len(skip_files)} file pairs")
print(f"Skipped {len(skip_files)} pairs")
print(f"Unique characters found: {''.join(sorted(all_chars))}")

Processing files (using 16 workers): 100%|██████████| 99325/99325 [23:34<00:00, 70.21it/s]

Processed 99325 file pairs
Skipped 0 pairs
Unique characters found: กขคฆงจฉชซฌญฎฏฐฑฒณดตถทธนบปผฝพฟภมยรฤลวศษสหฬอฮฯะัาำิีึืุูเแโใไ็่้๊๋์


# Save metadata

In [14]:
DEST_DIR = Path(DEST_DIR)

# Write character files
sorted_chars = sorted(all_chars)
with open(DEST_DIR / 'all_chars_unicode.txt', 'w') as f:
   f.write(''.join(c.encode('unicode_escape').decode('ascii') for c in sorted_chars))
   
with open(DEST_DIR / 'all_chars.txt', 'w') as f:
   f.write(''.join(sorted_chars))

# Resample, trim, and normalize audio

In [15]:
# Create destination directory if it doesn't exist
os.makedirs("../data/converted/thai-central-to-virtual-vctk/wav16_silence_trimmed", exist_ok=True)

# Copy all files from wav16 to wav16_silence_trimmed
src_dir = "../data/converted/thai-central-to-virtual-vctk/wav16"
dst_dir = "../data/converted/thai-central-to-virtual-vctk/wav16_silence_trimmed"

recursive_copy(src_dir, dst_dir)

In [16]:
# Resample all files in wav16_silence_trimmed to 16kHz
SAMPLE_RATE = 16000
NUM_RESAMPLE_THREADS = 8

resample_audios(
  input_folders=dst_dir,
  file_ext="flac",
  sample_rate=SAMPLE_RATE,
  n_jobs=NUM_RESAMPLE_THREADS
)

Resampling the audio files...
Found 99325 files...


100%|██████████| 99325/99325 [01:45<00:00, 945.91it/s] 


Done !


In [17]:
# Trim silence at the beginning and end of each audio file
trim_silence_with_vad(
  input_folder=dst_dir,
  file_extension="flac",
)

Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /home/ming/.cache/torch/hub/master.zip


Found 99325 .flac files to process


Processing files:  67%|██████▋   | 66842/99325 [2:22:10<50:10, 10.79it/s]  

> The file ../data/converted/thai-central-to-virtual-vctk/wav16_silence_trimmed/tc0332/thai-central_090647_mic1.flac probably does not have speech please check it !!


Processing files:  72%|███████▏  | 71637/99325 [2:32:21<58:01,  7.95it/s]  

> The file ../data/converted/thai-central-to-virtual-vctk/wav16_silence_trimmed/tc0510/thai-central_068154_mic1.flac probably does not have speech please check it !!


Processing files: 100%|██████████| 99325/99325 [3:30:28<00:00,  7.86it/s]  


Processing complete

Found 2 files with no speech. List saved to ../data/converted/thai-central-to-virtual-vctk/no_speech_files.txt


In [18]:
# Normalize the volume of all audio files to -27dB
normalize_audio_files(
  input_dir=dst_dir,
  extensions="flac",
  target_db=-27,
  sample_rate=16000,
)

Normalizing audio files: 100%|██████████| 99325/99325 [6:27:40<00:00,  4.27it/s]  


# New grouping for virtual speakerId

In [19]:
# Generate DVector using wavlm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained('microsoft/wavlm-base-plus-sv')
model = WavLMForXVector.from_pretrained('microsoft/wavlm-base-plus-sv').to(device)

if not os.path.exists('../data/converted/thai-central-to-virtual-vctk/dvectors.pth'):
    speaker_mapping = {}
    for speaker in tqdm(os.listdir(dst_dir)):
        for audio_file in os.listdir(os.path.join(dst_dir, speaker)):
            audio_path = os.path.join(dst_dir, speaker, audio_file)
            audio_input, sr = torchaudio.load(audio_path)
            audio_input = audio_input
            audio_array = audio_input.detach().cpu().numpy()
            with torch.no_grad():
                inputs = feature_extractor(audio_array, sampling_rate=16000, return_tensors="pt")
                inputs = {k: v.to(device) for k, v in inputs.items()}
                embeddings = model(**inputs).embeddings
                embeddings = torch.nn.functional.normalize(embeddings, dim=-1).cpu()
                if embeddings.isnan().any():
                    print(f"The embedding of {audio_file} is NaN")
                    continue
            speaker_mapping[audio_file] = {}
            speaker_mapping[audio_file]['embedding'] = embeddings
            speaker_mapping[audio_file]['name'] = speaker
    torch.save(speaker_mapping, '../data/converted/thai-central-to-virtual-vctk/dvectors.pth')
else:
    speaker_mapping = torch.load('../data/converted/thai-central-to-virtual-vctk/dvectors.pth')

  0%|          | 0/983 [00:00<?, ?it/s]/home/ming/.cache/pypoetry/virtualenvs/speech-dataset-converter-aRtuyZwp-py3.11/lib/python3.11/site-packages/torch/nn/functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
 97%|█████████▋| 953/983 [29:18<01:28,  2.96s/it]/home/ming/.cache/pypoetry/virtualenvs/speech-dataset-converter-aRtuyZwp-py3.11/lib/python3.11/site-packages/transformers/models/wavlm/modeling_wavlm.py:1833: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1831.)
  std_features.append(hidden_states[i, :length].std(dim=0))


The embedding of thai-central_131831_mic1.flac is NaN


100%|██████████| 983/983 [30:06<00:00,  1.84s/it]


In [20]:
BATCH_SIZE = 10000  # Adjust based on your system's RAM

def process_speaker(name, dvector_items, verbose=False):
    total_items = len(dvector_items)
    
    # Prepare embeddings
    dvector_embeddings = np.array([value['embedding'] for _, value in dvector_items])
    dvector_embeddings = dvector_embeddings.squeeze()
            
    cosine_distances_matrix = cosine_distances(dvector_embeddings, dvector_embeddings)

    dvector_clusterer = AgglomerativeClustering(
        metric="precomputed",
        linkage="average",
        distance_threshold=1-0.85,
        n_clusters=None
    ).fit(cosine_distances_matrix)

    labels = defaultdict(int)
    for label in dvector_clusterer.labels_:
        labels[label] += 1

    if verbose: print([(key, labels[key]) for key in sorted(labels, key=labels.get, reverse=True)])

    # if value < 10 change key to -1
    for i in range(total_items):
        if labels[dvector_clusterer.labels_[i]] < 10:
            dvector_clusterer.labels_[i] = -1

    dvector_labels = dvector_clusterer.labels_
    
    concat_labels = defaultdict(list)
    cluster_stats = defaultdict(lambda: {'file_count': 0})
    
    for i in range(total_items):
        key = dvector_items[i][0]  # Extract filename (key)
        cluster_key = f'{dvector_labels[i]}'
        concat_labels[cluster_key].append(key)
        cluster_stats[cluster_key]['file_count'] += 1

    gc.collect()

    return concat_labels

# Main processing loop
speaker_dict_dvector = defaultdict(list)

for k, v in speaker_mapping.items():
    name = v['name']
    speaker_dict_dvector[name].append((k, v))

global_speaker_dict = {}
for name in tqdm(speaker_dict_dvector, desc="Processing speakers"):
    concat_labels = process_speaker(name, speaker_dict_dvector[name])
    for cluster_key, cluster_files in concat_labels.items():
        if cluster_key == '-1':
            continue
        global_speaker_dict[f'v{name}_{cluster_key}'] = cluster_files

json.dump(global_speaker_dict, open('../data/converted/thai-central-to-virtual-vctk/virtual_speaker_mapping.json', 'w'), indent=4)

Processing speakers: 100%|██████████| 983/983 [05:51<00:00,  2.80it/s]


In [34]:
for speaker in tqdm(global_speaker_dict, desc="Processing speakers"):
    speaker_name = speaker
    original_speaker_name = speaker.split("_")[0][1:]
    speaker_audio_files = global_speaker_dict[speaker]
    speaker_text_files = [os.path.join(os.path.join(DEST_DIR, "txt"), f"{original_speaker_name}/{audio_file.split('.')[0]}.txt") for audio_file in speaker_audio_files]

    speaker_audio_dir = os.path.join("../data/converted/thai-central-to-virtual-vctk/wav16_silence_trimmed", speaker_name)
    os.makedirs(speaker_audio_dir, exist_ok=True)

    for audio_file in speaker_audio_files:
        src_audio_path = os.path.join(dst_dir, original_speaker_name, audio_file)
        dst_audio_path = os.path.join(speaker_audio_dir, audio_file)
        shutil.copy(src_audio_path, dst_audio_path)

    speaker_text_dir = os.path.join("../data/converted/thai-central-to-virtual-vctk/txt", speaker_name)
    os.makedirs(speaker_text_dir, exist_ok=True)
    
    for text_file in speaker_text_files:
        src_text_path = text_file
        dst_text_path = os.path.join(speaker_text_dir, os.path.basename(text_file))
        shutil.copy(src_text_path, dst_text_path)

Processing speakers: 100%|██████████| 995/995 [00:40<00:00, 24.69it/s]


In [ ]:
# remove folders with cv prefix in dst_dir
for speaker in os.listdir(dst_dir):
    if speaker.startswith('tc'):
        shutil.rmtree(os.path.join(dst_dir, speaker))

for speaker in os.listdir("../data/converted/thai-central-to-virtual-vctk/txt"):
    if speaker.startswith('tc'):
        shutil.rmtree(os.path.join("../data/converted/thai-central-to-virtual-vctk/txt", speaker))